# How LLM Tokenizers Work: A Hands-On Tutorial
## Understanding Text → Tokens → Generation

**What you'll learn:**
1. How tokenizers convert text into numbers (tokens)
2. Why different languages need different numbers of tokens
3. How chat templates format conversations for models
4. How models generate text one token at a time
5. How to control randomness in generation


## Setup (Multi-Platform: CUDA, Mac mini, CPU)

This notebook works on:
- **NVIDIA GPU** (CUDA) → Fastest ⚡
- **Apple Silicon** (M1/M2/M3 with MPS) → Fast 🍎
- **CPU** → Slower but works ⏱️

### Step 1: Install UV (Python package manager)

**Mac:**
```bash
brew install uv
```

**Windows/Linux:**
```bash
pip install uv
```

### Step 2: Setup your environment

Run these in terminal (from this directory):

```bash
# Create Python 3.11 environment with all dependencies
uv sync --python 3.11

# Activate virtual environment
source .venv/bin/activate

# Register Jupyter kernel
python -m ipykernel install --user --name=tokenisation-lab --display-name "Tokenisation Lab"
```

### Step 3: Verify GPU/Device Support

For **CUDA users:**
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

For **Mac users:**
- Just run setup above—MPS is auto-included

For **CPU users:**
- Works as-is, but will be slower

### Step 4: Open notebook
1. Restart your IDE (VS Code, Jupyter, etc)
2. Select kernel: **"Tokenisation Lab"** (top-right)
3. Run the device check cell below ↓

### Device Check (All Platforms)

We'll check what device is available:

In [25]:
import torch

print("=" * 70)
print("DEVICE CHECK - Multi-Platform Support")
print("=" * 70)
print(f"PyTorch version: {torch.__version__}")
print()

# Check all available devices
print("AVAILABLE DEVICES:")
print(f"  • CUDA (NVIDIA GPU): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"    - GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"    - CUDA Version: {torch.version.cuda}")

print(f"  • Apple Metal (MPS): {torch.backends.mps.is_available()}")
if torch.backends.mps.is_available():
    print(f"    - MPS Built: {torch.backends.mps.is_built()}")

print()

# Auto-select best device
if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ SELECTED: CUDA (NVIDIA GPU) - Fastest option!")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = "mps"
    print(f"✅ SELECTED: MPS (Apple Silicon) - Good performance!")
else:
    device = "cpu"
    print(f"⚠️  SELECTED: CPU - Slower, but will work fine")

print()
print(f"Recommended device: {device}")
print("=" * 70)

DEVICE CHECK - Multi-Platform Support
PyTorch version: 2.13.0

AVAILABLE DEVICES:
  • CUDA (NVIDIA GPU): False
  • Apple Metal (MPS): True
    - MPS Built: True

✅ SELECTED: MPS (Apple Silicon) - Good performance!

Recommended device: mps


In [26]:
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✅ All imports successful!")

# ============================================================
# MODEL CONFIGURATION (Low-param models for Mac mini)
# ============================================================
# We use GPT2 (125M params) - perfect for teaching!
# Other options:
#   - "gpt2" (125M) ← recommended, ~500MB
#   - "gpt2-medium" (355M) ← if you have more RAM
#   - "TinyLlama/TinyLlama-1.1B" ← balanced option, ~2.5GB

MODEL_ID = os.environ.get("MODEL_ID", "gpt2")
print(f"Model: {MODEL_ID}")
print("First run will download ~500MB (cached after)")
print()

✅ All imports successful!
Model: gpt2
First run will download ~500MB (cached after)



# SECTION 1: Strings → Token IDs
## How Tokenizers Convert Text to Numbers

### What is Tokenization?

The model doesn't read words. It reads **integers** (token IDs).

```
"Hello" → [50256, 18435]  (two token IDs)
```

A **tokenizer** is a lookup table:
- **Input:** Text (string)
- **Process:** Split text into tokens
- **Output:** List of integers (token IDs)

### Why This Matters

1. **Context window** — Models have a fixed token limit (e.g., 2048 tokens). More tokens = less space for history.
2. **Speed** — Generation is slower with more tokens.
3. **Language efficiency** — Languages like English use fewer tokens; other languages use more.

In [27]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Display basic info
print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size:,} unique tokens")
print(f"EOS token (end-of-sequence): {repr(tokenizer.eos_token)} (ID: {tokenizer.eos_token_id})")
print(f"PAD token (padding): {repr(tokenizer.pad_token)} (ID: {tokenizer.pad_token_id})")

Tokenizer: GPT2Tokenizer
Vocabulary size: 50,257 unique tokens
EOS token (end-of-sequence): '<|endoftext|>' (ID: 50256)
PAD token (padding): None (ID: None)


### Let's Tokenize Some Text

We'll convert text to token IDs, then decode them back:

In [28]:
# Example 1: Simple English
text1 = "Hello, how are you?"
token_ids = tokenizer.encode(text1, add_special_tokens=False)

print("TEXT:", text1)
print("TOKEN IDs:", token_ids)
print("NUM TOKENS:", len(token_ids))
print()

# Example 2: Longer sentence
text2 = "I love learning about how language models work!"
token_ids2 = tokenizer.encode(text2, add_special_tokens=False)

print("TEXT:", text2)
print("TOKEN IDs:", token_ids2)
print("NUM TOKENS:", len(token_ids2))

TEXT: Hello, how are you?
TOKEN IDs: [15496, 11, 703, 389, 345, 30]
NUM TOKENS: 6

TEXT: I love learning about how language models work!
TOKEN IDs: [40, 1842, 4673, 546, 703, 3303, 4981, 670, 0]
NUM TOKENS: 9


### Inspect Each Token

What does each token ID actually represent? Let's see:

In [29]:
def show_tokens(text, max_tokens=50):
    """Display each token in a nice table"""
    ids = tokenizer.encode(text, add_special_tokens=False)
    
    df = pd.DataFrame({
        "Position": range(len(ids)),
        "Token ID": ids,
        "Token Text": [repr(tokenizer.decode([id_])) for id_ in ids]
    })
    
    print(f"Text: {text}\n")
    print(f"Total tokens: {len(ids)}\n")
    return df.head(max_tokens)

# Example
result = show_tokens("Hello, how are you?")
result

Text: Hello, how are you?

Total tokens: 6



,Position,Token ID,Token Text
0,0,15496,'Hello'
1,1,11,"','"
2,2,703,' how'
3,3,389,' are'
4,4,345,' you'
5,5,30,'?'


### Challenge: Language Efficiency

Different languages use different numbers of tokens! Try comparing:

**Same idea in different languages:**
- English: "Hello, how are you?" 
- Spanish: "Hola, ¿cómo estás?"
- Hindi: "नमस्ते, आप कैसे हैं?"

Why? The tokenizer was trained mostly on English text, so English is more "efficient" (fewer tokens per word).

In [30]:
# Compare tokenization across languages
examples = [
    ("English", "Hello, how are you?"),
    ("Spanish", "Hola, ¿cómo estás?"),
    ("French", "Bonjour, comment allez-vous?"),
    ("Hindi", "नमस्ते, आप कैसे हैं?"),
    ("Telugu", "హలో, మీరు ఎలా ఉన్నారు?"),
    ("Kannada", "ಹಲೋ, ನೀವು ಹೇಗಿದ್ದೀರಿ?"),
]

rows = []
for language, text in examples:
    ids = tokenizer.encode(text, add_special_tokens=False)
    chars_per_token = round(len(text) / len(ids), 2) if len(ids) > 0 else 0
    rows.append({
        "Language": language,
        "Text Length (chars)": len(text),
        "Num Tokens": len(ids),
        "Chars/Token": chars_per_token
    })

df_comparison = pd.DataFrame(rows)
print("\n📊 LANGUAGE EFFICIENCY COMPARISON\n")
print(df_comparison.to_string(index=False))
print("\n" + "="*70)
print("💡 KEY INSIGHTS:")
print("  • English is MOST efficient (more chars per token)")
print("  • Indian languages (Hindi, Telugu, Kannada) need MORE tokens")
print("  • Why? Tokenizer trained mostly on English text")
print("  • Impact: Same idea = more tokens in Telugu = uses more context space")
print("="*70)


📊 LANGUAGE EFFICIENCY COMPARISON

Language  Text Length (chars)  Num Tokens  Chars/Token
 English                   19           6         3.17
 Spanish                   18          11         1.64
  French                   28          10         2.80
   Hindi                   20          32         0.62
  Telugu                   22          56         0.39
 Kannada                   21          55         0.38

💡 KEY INSIGHTS:
  • English is MOST efficient (more chars per token)
  • Indian languages (Hindi, Telugu, Kannada) need MORE tokens
  • Why? Tokenizer trained mostly on English text
  • Impact: Same idea = more tokens in Telugu = uses more context space


# SECTION 2: Chat Templates
## How Conversations are Formatted for Models

### The Problem

Base language models just predict the next token. They have NO understanding of "user" or "assistant" roles.

```
Input:  "What is 2+2?"
Output: "What is 2+2? What is 2+2? What is 2+2?..." (just repeats!)
```

### The Solution: Chat Templates

We format conversations in a special way so the model learns the pattern.

**Example format:**
```
[SYSTEM]: You are a helpful assistant
[USER]: What is 2+2?
[ASSISTANT]: 
```

After training on millions of conversations in this format, the model learns:
- After `[USER]:`, it should answer the question
- After `[ASSISTANT]:`, it should provide the reply

### Why Different Models Have Different Formats

Different models are trained on different conversation patterns:

- **GPT2** (base model): Needs manual formatting like above
- **ChatGPT/GPT-3.5-Turbo**: Uses `<|im_start|>` `<|im_end|>`
- **Llama-Chat**: Uses `[INST]` `[/INST]`
- **Qwen**: Uses `<|im_start|>` `<|im_end|>`

Mixing them up = garbage output! Because the model doesn't recognize the pattern.

In [7]:
# Since GPT2 doesn't have a built-in chat template, we'll create one manually
# This shows what chat formatting looks like for a base model

messages = [
    {"role": "system", "content": "You are a helpful math tutor."},
    {"role": "user", "content": "What is 2+2?"},
]

# Manually format into a simple chat template (common pattern for base models)
chat_text = ""
for msg in messages:
    role = msg["role"].upper()
    content = msg["content"]
    chat_text += f"[{role}]: {content}\n"

# Add the assistant turn (where the model will continue)
chat_text += "[ASSISTANT]: "

print("📋 FORMATTED CHAT TEMPLATE (Manual Format):\n")
print(chat_text)
print("\n" + "="*60)
print("💡 How it works:")
print("  • We format each message with [ROLE]: content")
print("  • We end with [ASSISTANT]: so the model knows it's its turn")
print("  • The model learned this pattern from training data")
print("  • Different models use different formats!")
print("="*60)

📋 FORMATTED CHAT TEMPLATE (Manual Format):

[SYSTEM]: You are a helpful math tutor.
[USER]: What is 2+2?
[ASSISTANT]: 

💡 How it works:
  • We format each message with [ROLE]: content
  • We end with [ASSISTANT]: so the model knows it's its turn
  • The model learned this pattern from training data
  • Different models use different formats!


In [8]:
# Now tokenize that formatted text
chat_ids = tokenizer.encode(chat_text, add_special_tokens=False)

print("📊 SAME TEXT AS TOKEN IDs:\n")
print(f"Total tokens: {len(chat_ids)}\n")

df_chat = pd.DataFrame({
    "Position": range(len(chat_ids)),
    "Token ID": chat_ids,
    "Token Text": [repr(tokenizer.decode([id_])) for id_ in chat_ids]
})

print(df_chat.to_string(index=False))

print("\n" + "="*60)
print("👆 Notice:")
print("  • Each word/punctuation is a token")
print("  • The format tells the model: 'This is a conversation'")
print("  • Model predicts what [ASSISTANT] should say next")
print("="*60)

📊 SAME TEXT AS TOKEN IDs:

Total tokens: 28

 Position  Token ID Token Text
        0        58        '['
        1     23060       'SY'
        2     25361     'STEM'
        3      5974       ']:'
        4       921     ' You'
        5       389     ' are'
        6       257       ' a'
        7      7613 ' helpful'
        8     10688    ' math'
        9     48680   ' tutor'
       10        13        '.'
       11       198       '\n'
       12        58        '['
       13     29904     'USER'
       14      5974       ']:'
       15      1867    ' What'
       16       318      ' is'
       17       362       ' 2'
       18        10        '+'
       19        17        '2'
       20        30        '?'
       21       198       '\n'
       22        58        '['
       23     10705      'ASS'
       24      8808      'IST'
       25      8643      'ANT'
       26      5974       ']:'
       27       220        ' '

👆 Notice:
  • Each word/punctuation is a token
  • The 

# SECTION 3: Text Generation
## How Models Generate Text Token-by-Token

### The Generation Loop

Every response is built one token at a time:

```
Step 1: Look at input, predict next token → "The"
Step 2: Append "The", look at input+output, predict → "cat"
Step 3: Append "cat", predict → "sat"
Step 4: Append "sat", predict → "."
(repeat until done)
```

**That's it.** There's no "write a sentence" function. Just a loop.

### How It Works

1. **Tokenizer:** Converts prompt to token IDs
2. **Model:** Takes token IDs, outputs probability for each word
3. **Sampler:** Picks the most likely word (or random based on temperature)
4. **Repeat:** Add new token, predict next one

Let's see this in action!

### Step 1: Load the Model

This loads the actual model weights into memory (or GPU). First run takes ~30 seconds.

In [9]:
# Auto-detect best device and data type
print("LOADING MODEL SETUP")
print("="*70)

# Device selection
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
    device_name = "CUDA (NVIDIA GPU)"
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = "mps"
    dtype = torch.float32
    device_name = "MPS (Apple Silicon)"
    print(f"✅ Device: Apple Silicon (M1/M2/M3)")
else:
    device = "cpu"
    dtype = torch.float32
    device_name = "CPU"
    print(f"⚠️  Device: CPU (slower, but will work)")

print()
print(f"Loading model: {MODEL_ID}")
print(f"Device: {device_name} | Dtype: {dtype}")
print("(This takes ~30-60 seconds on first run)\n")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,  # Updated: use 'dtype' instead of deprecated 'torch_dtype'
    trust_remote_code=True,
).to(device)  # Move to device directly

model.eval()

print(f"✅ Model loaded successfully on {device_name}")
print("="*70)

LOADING MODEL SETUP
✅ Device: Apple Silicon (M1/M2/M3)

Loading model: gpt2
Device: MPS (Apple Silicon) | Dtype: torch.float32
(This takes ~30-60 seconds on first run)



model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded successfully on MPS (Apple Silicon)


### Step 2: One Token at a Time

Let's generate text and watch what the model predicts at each step:

In [12]:
@torch.inference_mode()
def generate_step_by_step(prompt, max_steps=20, temperature=0.7, show_top_k=5):
    """Generate text one token at a time, showing predictions"""
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    print("="*80)
    print(f"PROMPT: {prompt}\n")
    print("GENERATION (one token at a time):")
    print("="*80 + "\n")
    
    rows = []
    
    for step in range(max_steps):
        # Get model prediction
        with torch.no_grad():
            outputs = model(input_ids=generated)
        
        # Only look at the last position
        next_logits = outputs.logits[0, -1, :]
        probs = torch.softmax(next_logits, dim=-1)
        
        # Apply temperature
        if temperature > 0:
            scaled_probs = torch.softmax(next_logits / temperature, dim=-1)
            next_id = torch.multinomial(scaled_probs, num_samples=1)
        else:
            next_id = torch.argmax(next_logits, dim=-1, keepdim=True)
        
        token_id = next_id.item()
        token_text = tokenizer.decode([token_id])
        token_prob = probs[token_id].item()
        
        # Get top-k predictions
        top_probs, top_ids = torch.topk(probs, show_top_k)
        top_text = " | ".join(
            f"{repr(tokenizer.decode([tid.item()]))} ({tp.item():.2%})"
            for tid, tp in zip(top_ids, top_probs)
        )
        
        rows.append({
            "Step": step + 1,
            "Generated": repr(token_text),
            "Confidence": f"{token_prob:.1%}",
            "Top Predictions": top_text
        })
        
        print(f"Step {step + 1}: {repr(token_text)} (confidence: {token_prob:.1%})")
        
        # Append new token
        generated = torch.cat([generated, next_id.unsqueeze(0)], dim=-1)
        
        # Stop if end of sequence
        if token_id == tokenizer.eos_token_id:
            print("\n[Model stopped (EOS token)]")
            break
    
    print("\n" + "="*80)
    print(f"FULL OUTPUT: {tokenizer.decode(generated[0])}")
    print("="*80)
    
    return pd.DataFrame(rows)

# Generate text
prompt = "The future of artificial intelligence is"
df_generation = generate_step_by_step(prompt, max_steps=15, temperature=0.7)

PROMPT: The future of artificial intelligence is

GENERATION (one token at a time):

Step 1: ' unclear' (confidence: 1.8%)
Step 2: '.' (confidence: 39.3%)
Step 3: ' A' (confidence: 2.1%)
Step 4: ' lot' (confidence: 4.5%)
Step 5: ' of' (confidence: 76.6%)
Step 6: ' projects' (confidence: 0.1%)
Step 7: ' have' (confidence: 14.2%)
Step 8: ' been' (confidence: 30.6%)
Step 9: ' worked' (confidence: 0.4%)
Step 10: ' on' (confidence: 85.8%)
Step 11: ' in' (confidence: 7.2%)
Step 12: ' recent' (confidence: 12.9%)
Step 13: ' years' (confidence: 92.4%)
Step 14: ',' (confidence: 40.4%)
Step 15: ' but' (confidence: 38.8%)

FULL OUTPUT: The future of artificial intelligence is unclear. A lot of projects have been worked on in recent years, but


In [13]:
# Show the generation table
print("\n📊 GENERATION TRACE:\n")
with pd.option_context("display.max_colwidth", None, "display.width", 200):
    print(df_generation.to_string(index=False))


📊 GENERATION TRACE:

 Step   Generated Confidence                                                                                    Top Predictions
    1  ' unclear'       1.8%            ' uncertain' (7.66%) | ' in' (6.94%) | ' not' (4.41%) | ' a' (4.07%) | ' still' (3.45%)
    2         '.'      39.3%                          '.' (39.29%) | ',' (32.05%) | '."' (5.77%) | ',"' (3.62%) | ' at' (2.61%)
    3        ' A'       2.1%                     '\n' (8.71%) | ' But' (8.08%) | ' The' (7.35%) | ' It' (4.57%) | ' In' (4.25%)
    4      ' lot'       4.5%           ' new' (6.56%) | ' few' (5.38%) | ' lot' (4.49%) | ' number' (4.34%) | ' recent' (3.50%)
    5       ' of'      76.6%             ' of' (76.58%) | ' has' (7.80%) | ' depends' (4.88%) | ' is' (3.00%) | ' will' (1.56%)
    6 ' projects'       0.1%        ' the' (16.31%) | ' people' (9.96%) | ' research' (5.58%) | ' work' (5.27%) | ' it' (4.56%)
    7     ' have'      14.2%                 ' are' (23.32%) | ' have' (14.20%) | 

### Step 3: Controlling Randomness with Temperature

Models don't always pick the most likely token. We can control how "creative" or "predictable" they are.

**Temperature:**
- `temperature = 0` → Always pick most likely (deterministic)
- `temperature = 1` → Normal randomness
- `temperature > 1` → More random, creative
- `temperature < 1` → More conservative

Let's see the difference:

In [14]:
@torch.inference_mode()
def generate_with_temperature(prompt, temperature=0.7, max_new_tokens=20):
    """Simple generation using model.generate()"""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature if temperature > 0 else 1.0,
        do_sample=temperature > 0,  # Sample if temperature > 0
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    return tokenizer.decode(output[0])

# Compare different temperatures
prompt = "Machine learning is"
temperatures = [0.3, 0.7, 1.5]

print("🌡️  TEMPERATURE COMPARISON\n")
print("="*80 + "\n")

for temp in temperatures:
    result = generate_with_temperature(prompt, temperature=temp, max_new_tokens=15)
    print(f"Temperature = {temp}:")
    print(f"Output: {result.strip()}\n")
    print("-"*80 + "\n")

🌡️  TEMPERATURE COMPARISON


Temperature = 0.3:
Output: Machine learning is a great way to learn about the world around you.

The next

--------------------------------------------------------------------------------

Temperature = 0.7:
Output: Machine learning is not going to be a new thing.

The problem is that there

--------------------------------------------------------------------------------

Temperature = 1.5:
Output: Machine learning is still ongoing by most standards, even as research continues, and its findings are

--------------------------------------------------------------------------------



### Step 4: Watch the Detailed Generation Trace

This shows **every token the model considers** at each step, with probabilities and confidence scores:

In [17]:
@torch.inference_mode()
def generate_one_token_at_a_time(input_ids, steps=20, temperature=0.0, top_k_display=5):
    """
    Generate text one token at a time, showing detailed trace.
    
    Returns:
    - generated: full sequence of token IDs
    - DataFrame: trace with step, chosen token, confidence, and top candidates
    """
    generated = input_ids.clone()
    rows = []

    for step in range(steps):
        # Get model output
        outputs = model(input_ids=generated)
        next_logits = outputs.logits[:, -1, :]  # Last position only
        probs = torch.softmax(next_logits, dim=-1)

        # Choose next token (greedy or sampled)
        if temperature and temperature > 0:
            scaled_probs = torch.softmax(next_logits / temperature, dim=-1)
            next_id = torch.multinomial(scaled_probs, num_samples=1)
        else:
            next_id = torch.argmax(next_logits, dim=-1, keepdim=True)

        token_id = next_id.item()
        token_text = tokenizer.decode([token_id])
        chosen_prob = probs[0, token_id].item()

        # Get top-k candidates
        top_probs, top_ids = torch.topk(probs[0], top_k_display)
        top_candidates = " | ".join(
            f"{repr(tokenizer.decode([tid.item()]))} {tp.item():.3f}{'*' if tid.item() == token_id else ''}"
            for tid, tp in zip(top_ids, top_probs)
        )

        rows.append({
            "step": step + 1,
            "chosen_token": repr(token_text),
            "confidence": f"{chosen_prob:.1%}",
            f"top {top_k_display} candidates (* = chosen)": top_candidates,
        })

        # Append new token and continue
        generated = torch.cat([generated, next_id], dim=-1)

        # Stop if end of sequence
        if token_id == tokenizer.eos_token_id:
            print(f"\n✅ Stopped at step {step + 1} (EOS token)")
            break

    return generated, pd.DataFrame(rows)


# Generate with detailed trace
prompt = "The future of AI is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
generated, trace = generate_one_token_at_a_time(input_ids, steps=20, temperature=0.0)

# Show the trace
print(f"\n📊 GENERATION TRACE FOR: '{prompt}'\n")
print(trace.to_string(index=False))

# Show full generated text
print(f"\n📝 FULL OUTPUT:\n")
print(tokenizer.decode(generated[0]))


📊 GENERATION TRACE FOR: 'The future of AI is'

 step chosen_token confidence                                                                     top 5 candidates (* = chosen)
    1 ' uncertain'       6.1%                    ' uncertain' 0.061* | ' in' 0.059 | ' not' 0.045 | ' a' 0.040 | ' still' 0.036
    2          '.'      34.7%                                   '.' 0.347* | ',' 0.322 | ',"' 0.040 | ' and' 0.038 | '."' 0.034
    3       ' The'       7.1%                             ' The' 0.071* | '\n' 0.066 | ' But' 0.065 | ' It' 0.059 | ' We' 0.047
    4    ' future'       3.7%       ' future' 0.037* | ' question' 0.021 | ' most' 0.019 | ' world' 0.018 | ' technology' 0.018
    5        ' of'      65.8%                        ' of' 0.658* | ' is' 0.126 | ' will' 0.029 | ' may' 0.020 | ' could' 0.010
    6        ' AI'      19.4%           ' AI' 0.194* | ' the' 0.062 | ' artificial' 0.046 | ' human' 0.041 | ' computing' 0.028
    7        ' is'      50.0%                           

### Advanced: How Entropy Reveals Uncertainty

**What is entropy?** It measures how uncertain the model is:
- **High entropy** = Model is confused (many likely tokens)
- **Low entropy** = Model is confident (one clear winner)

This next demo finds the most uncertain step in generation, then shows how temperature and top_p affect sampling at that uncertain moment:

In [16]:
# Step 1: Find the step with highest entropy (most uncertainty)
search_steps = 15  # How many steps to search
ctx = tokenizer.encode(prompt, return_tensors="pt").to(device)
best_logits, best_entropy, best_step = None, -1.0, 0

print("🔍 Searching for highest entropy step...\n")

with torch.inference_mode():
    for step in range(search_steps):
        out = model(input_ids=ctx)
        logits = out.logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        
        # Calculate entropy: how uncertain is the model?
        entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum().item()
        
        # Keep track of the step with highest entropy
        if entropy > best_entropy:
            best_logits, best_entropy, best_step = logits, entropy, step
        
        # Greedy continuation
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        ctx = torch.cat([ctx, next_id], dim=-1)
        
        print(f"Step {step}: entropy = {entropy:.3f}")

print(f"\n✅ Found highest entropy at step {best_step} (entropy: {best_entropy:.3f})")
print(f"At this step, the model is most UNCERTAIN about what comes next!\n")

# Step 2: Sample from this uncertain moment with different temperature/top_p
print("="*70)
print("SAMPLING FROM THE MOST UNCERTAIN MOMENT")
print("="*70 + "\n")

settings = [
    (0.3, 0.95, "Conservative (predictable)"),
    (0.7, 0.90, "Balanced (normal)"),
    (1.0, 0.90, "Standard"),
    (1.3, 0.95, "Creative"),
    (2.0, 1.00, "Wild (very random)"),
]

rows = []

for temperature, top_p, description in settings:
    # Apply temperature
    scaled_logits = best_logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    
    # Apply top_p filtering
    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        keep = cumulative <= top_p
        keep[..., 0] = True  # Always keep at least the top token
        sorted_probs = (sorted_probs * keep) / (sorted_probs * keep).sum(dim=-1, keepdim=True)
        filtered = torch.zeros_like(probs)
        filtered.scatter_(1, sorted_idx, sorted_probs)
        probs = filtered
    
    # Sample 24 times
    sample_ids = torch.multinomial(probs, num_samples=24, replacement=True).squeeze(0).tolist()
    samples = [repr(tokenizer.decode([i])) for i in sample_ids]
    
    rows.append({
        "Temperature": temperature,
        "Top-p": top_p,
        "Description": description,
        "Unique Tokens (24 draws)": len(set(samples)),
        "Samples": " | ".join(samples[:8]) + "..."  # Show first 8
    })

df_sampling = pd.DataFrame(rows)
print(df_sampling.to_string(index=False))

print("\n" + "="*70)
print("💡 WHAT THIS SHOWS:")
print("  • Low temp (0.3) = Same token repeated → BORING")
print("  • Normal temp (0.7-1.0) = Variety, but sensible → GOOD")
print("  • High temp (2.0) = Wild variety, sometimes gibberish → CREATIVE")
print("  • Top-p controls WHEN to stop exploring alternatives")
print("="*70)

Task was destroyed but it is pending!
task: <Task pending name='Task-153' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/sailokeshk/Documents/TTI Vault/TokenstoIntelligence/environment_setup/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-154' coro=<Kernel.shell_main() running at /Users/sailokeshk/Documents/TTI Vault/TokenstoIntelligence/environment_setup/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/sailokeshk/Documents/TTI Vault/TokenstoIntelligence/environment_setup/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/sailokeshk/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending

🔍 Searching for highest entropy step...

Step 0: entropy = 5.510
Step 1: entropy = 6.096
Step 2: entropy = 4.925
Step 3: entropy = 4.069
Step 4: entropy = 2.785
Step 5: entropy = 6.236
Step 6: entropy = 4.617
Step 7: entropy = 5.567
Step 8: entropy = 4.367
Step 9: entropy = 2.062
Step 10: entropy = 0.658
Step 11: entropy = 1.783
Step 12: entropy = 4.497
Step 13: entropy = 3.557
Step 14: entropy = 5.007

✅ Found highest entropy at step 5 (entropy: 6.236)
At this step, the model is most UNCERTAIN about what comes next!

SAMPLING FROM THE MOST UNCERTAIN MOMENT

 Temperature  Top-p                Description  Unique Tokens (24 draws)                                                                                                                 Samples
         0.3   0.95 Conservative (predictable)                         7                        ' building' | ' the' | ' solving' | ' learning' | ' learning' | ' the' | ' the' | ' building'...
         0.7   0.90          Balanced (normal)   

# BONUS: Try It Yourself!

Now it's time to experiment. Use the cells below to:

1. **Tokenize different languages** → Show why some languages are less efficient
2. **Create different prompts** → See how context affects generation
3. **Adjust temperature** → Find the right balance for different tasks
4. **Create conversations** → Build multi-turn chat examples

Have fun exploring! 🚀

### Activity 1: Tokenize Your Own Text

Try different inputs and see how many tokens they take:

In [ ]:
# TODO: Change these texts to whatever you want!
my_texts = [
    "Hello world!",
    "Your text here",
    "Try a long sentence to see how many tokens it takes",
]

print("YOUR TOKENIZATION EXPERIMENT:\n")
for text in my_texts:
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"Text: {repr(text)}")
    print(f"Tokens: {ids}")
    print(f"Count: {len(ids)} tokens\n")

### Activity 2: Generate Text with Different Prompts

Create different prompts and see what the model generates:

In [ ]:
# TODO: Try your own prompts!
my_prompts = [
    "Python is a",
    "The best way to learn programming is",
    "In the year 2030, AI will",
]

print("YOUR GENERATION EXPERIMENT:\n")
print("="*80 + "\n")

for prompt in my_prompts:
    result = generate_with_temperature(prompt, temperature=0.8, max_new_tokens=20)
    print(f"Prompt: {repr(prompt)}")
    print(f"Output: {result.strip()}")
    print("\n" + "-"*80 + "\n")

# Summary: What You've Learned

## 🎯 The Big Picture

```
TEXT INPUT
    ↓
[Tokenizer: Text → Token IDs]
    ↓
[Model: Predict next token probability]
    ↓
[Sampler: Pick best token (or random)]
    ↓
[Repeat until done]
    ↓
GENERATED TEXT
```

## 📚 Key Concepts

1. **Tokenization:** Models don't see words—they see integers
2. **Efficiency:** Different languages need different numbers of tokens
3. **Chat Templates:** Special formatting teaches models how to chat
4. **Generation:** Always one token at a time, in a loop
5. **Temperature:** Controls creativity vs consistency

## 🚀 Next Steps

- Try longer generations
- Experiment with different prompts
- Fine-tune a model on your own data
- Build a chatbot using these concepts
- Read the transformer architecture paper

---

**Congratulations!** You now understand how the core of modern LLMs work. 🎉